# 10.05 · Eval tooling & CI

**Chapter:** Evaluation · **Milestone:** M5

## Goals
- [ ] Turn your checks into a **regression suite** that runs on every change
- [ ] Use three tools on the same task: plain pytest, **promptfoo** (config-driven), **Inspect AI** (Python, research-grade)
- [ ] Wire evals into GitHub Actions; know the hosted platforms

## Resources
- [promptfoo docs](https://www.promptfoo.dev/docs/intro/) — free/OSS
- [Inspect AI docs](https://inspect.aisi.org.uk/) — free/OSS (UK AI Security Institute)
- [Braintrust](https://www.braintrust.dev/docs) · [LangSmith evaluation](https://docs.smith.langchain.com/evaluation) · [Langfuse datasets](https://langfuse.com/docs/evaluation/overview) — free tiers

| Tool | Style | Best for |
|---|---|---|
| **pytest + your code** | tests in Python | full control, no new tool, runs in any CI |
| **promptfoo** | YAML config, CLI, web viewer, red-teaming | comparing prompts × models fast; non-Python teams |
| **Inspect AI** | Python tasks: dataset → solver → scorer; logs viewer | agent/benchmark evals, sandboxed tools |
| **DeepEval** | pytest plugin with LLM metrics | RAG/LLM metrics as unit tests |
| **Braintrust / LangSmith / Langfuse / Phoenix** | hosted datasets, experiments, dashboards | teams, history, linking evals to traces |

The task for all three: ticket routing from 06.01 (`chapters/06-.../tickets.json`).

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from aieng import DATA
from aieng.llm import client

c, MODEL = client()
LOCAL = c.base_url.host == "localhost"
TICKETS = json.loads(Path("../06-prompt-and-context-engineering/tickets.json").read_text())
SYSTEM = ("Classify the support ticket into exactly one of: billing, bug, feature_request, account, other. "
          "account = login/password/email/deletion; billing = charges/invoices/refunds. Answer with the category only.")
EVAL_DIR = DATA / "evals"; EVAL_DIR.mkdir(exist_ok=True)

## 1. Plain pytest
A test file is the most portable eval harness there is. Threshold-based: the suite fails if accuracy drops below a bar.

In [ ]:
test_file = EVAL_DIR / "test_routing.py"
test_file.write_text(f'''
import json
from pathlib import Path
from aieng.llm import client

TICKETS = json.loads(Path({str(Path("../06-prompt-and-context-engineering/tickets.json").resolve())!r}).read_text())
SYSTEM = {SYSTEM!r}
c, MODEL = client()
EXTRA = {{"reasoning_effort": "none"}} if c.base_url.host == "localhost" else {{}}

def classify(text):
    r = c.chat.completions.create(model=MODEL, temperature=0, max_tokens=10, **EXTRA,
                                  messages=[{{"role": "system", "content": SYSTEM}}, {{"role": "user", "content": text}}])
    return r.choices[0].message.content.strip().lower()

def test_accuracy_above_threshold():
    preds = [classify(t["text"]) for t in TICKETS]
    acc = sum(p == t["label"] for p, t in zip(preds, TICKETS)) / len(TICKETS)
    print(f"accuracy={{acc:.2f}}")
    assert acc >= 0.7, f"routing accuracy regressed: {{acc:.2f}}"

def test_output_is_a_valid_label():
    labels = {{"billing", "bug", "feature_request", "account", "other"}}
    assert all(classify(t["text"]) in labels for t in TICKETS[:5])
''')
r = subprocess.run(["uv", "run", "--with", "pytest", "pytest", "-q", "-s", str(test_file)], capture_output=True, text=True, cwd="../..")
print(r.stdout[-800:], r.stderr[-300:])

## 2. promptfoo
Declarative: prompts × providers × test cases × assertions. Runs via `npx` (Node.js needed: `brew install node`).
Here the provider is the same OpenAI-compatible endpoint as everything else. Assertions mix exact checks with an `llm-rubric` judge.

In [ ]:
provider = {"id": f"openai:chat:{MODEL}", "config": {"apiBaseUrl": str(c.base_url).rstrip("/"), "apiKey": c.api_key, "temperature": 0,
                                                     **({"passthrough": {"reasoning_effort": "none"}} if LOCAL else {})}}
config = {
    "description": "ticket routing",
    "prompts": [json.dumps([{"role": "system", "content": SYSTEM}, {"role": "user", "content": "{{text}}"}])],
    "providers": [provider],
    "defaultTest": {"options": {"provider": provider}},  # the llm-rubric judge uses the same model (use a stronger one for real)
    "tests": [{"vars": {"text": t["text"]}, "assert": [{"type": "equals", "value": t["label"], "transform": "output.trim().toLowerCase()"}]}
              for t in TICKETS[:12]]
              + [{"vars": {"text": "Delete my account NOW or I'll sue"},
                  "assert": [{"type": "icontains", "value": "account"},
                             {"type": "llm-rubric", "value": "The output is a single category word with no extra text"}]}],
}
cfg_path = EVAL_DIR / "promptfooconfig.json"
cfg_path.write_text(json.dumps(config, indent=1))

env = {**os.environ, "PROMPTFOO_DISABLE_TELEMETRY": "1", "PROMPTFOO_DISABLE_UPDATE": "1"}
r = subprocess.run(["npx", "-y", "promptfoo@0.124.1", "eval", "-c", str(cfg_path), "--no-progress-bar", "--no-cache", "-o", str(EVAL_DIR / "promptfoo-results.json")],
                   capture_output=True, text=True, env=env, timeout=900)
print(r.stdout[-1500:] or r.stderr[-1500:])

`npx promptfoo@0.124.1 view` opens a web UI to browse results side by side. Add more `providers` (e.g. Groq + Gemini) to compare models
on the same tests in one run — that's promptfoo's sweet spot. `promptfoo redteam` generates adversarial tests (chapter 11).

## 3. Inspect AI
A task = dataset + solver (how the model is called) + scorer. Logs every sample; `inspect view` browses them.
Current Inspect needs `openai>=3.4`, which this repo's `litellm` pin blocks — so we run it in an **isolated** env (`uv run --isolated`), the
usual fix when an eval tool's dependencies clash with your app's. The task lives in a file; the CLI runs it.

In [ ]:
task_file = EVAL_DIR / "inspect_task.py"
task_file.write_text(f'''
import json
from pathlib import Path
from inspect_ai import Task, task
from inspect_ai.dataset import Sample
from inspect_ai.scorer import match
from inspect_ai.solver import generate, system_message

TICKETS = json.loads(Path({str(Path("../06-prompt-and-context-engineering/tickets.json").resolve())!r}).read_text())

@task
def ticket_routing():
    return Task(dataset=[Sample(input=t["text"], target=t["label"]) for t in TICKETS],
                solver=[system_message({SYSTEM!r}), generate()],
                scorer=match(location="any", ignore_case=True))
''')

cmd = ["uv", "run", "-q", "--isolated", "--no-project", "--python", "3.12", "--with", "inspect-ai", "--with", "openai>=3.4",
       "inspect", "eval", str(task_file), "--model", f"openai-api/aieng/{MODEL}",  # any OpenAI-compatible endpoint via AIENG_* env vars
       "--log-dir", str(EVAL_DIR / "inspect-logs"), "--display", "plain", "--temperature", "0", "--max-tokens", "20"]
cmd += ["--reasoning-effort", "none"] if LOCAL else []
r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800,
                   env={**os.environ, "AIENG_BASE_URL": str(c.base_url), "AIENG_API_KEY": c.api_key})
print((r.stdout + r.stderr)[-700:])

`uv run --isolated --with inspect-ai inspect view --log-dir data/evals/inspect-logs` opens the log viewer. Inspect's real strength is agentic evals: tools, multi-turn
solvers and Docker sandboxes, and a large library of ready benchmarks (`inspect_evals`).

## 4. In CI (GitHub Actions)
Run fast, cheap evals on every PR; the expensive full suite nightly or before releases. Use a hosted model in CI (no Ollama there) with the key
as a repository secret. Example `.github/workflows/evals.yml`:
```yaml
name: evals
on: [pull_request]
jobs:
  evals:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v6
      - run: uv sync
      - run: uv run --with pytest pytest -q evals/
        env:
          AIENG_PROVIDER: groq
          GROQ_API_KEY: ${{ secrets.GROQ_API_KEY }}
      # or: npx promptfoo@latest eval -c evals/promptfooconfig.yaml --fail-on-error
```
Tips: cache LLM responses for unchanged prompts (05.05), pin model versions, keep thresholds slightly below current scores (LLM noise),
and post the score diff as a PR comment.

## Exercises
1. Add a second provider (Groq or Gemini) to the promptfoo config and compare both models in one run. Open `promptfoo view`.
2. Write an Inspect task for 10.02's synthetic RAG set: solver = a custom solver calling `rag.answer`, scorer = `model_graded_fact()`.
3. Put `test_routing.py` in an `evals/` folder of your GitHub repo with the workflow above and a `GROQ_API_KEY` secret. Break the prompt on purpose and watch CI fail.

In [ ]:
# your code here